In [1]:
# =============================================================
# CELLULE 1 — IMPORTS ET CONFIGURATION
# =============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Seed pour la reproductibilité
SEED = 42
np.random.seed(SEED)

# Style des graphiques
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"]      = 12

# Options d'affichage pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows",    100)
pd.set_option("display.precision",   2)

print("✓ Environnement configuré avec succès.")

✓ Environnement configuré avec succès.


In [2]:
# =============================================================
# CELLULE 2 — CHARGEMENT DU DATASET NETTOYÉ
# =============================================================

# On charge data_clean.csv (résultat du NB01).
# RÈGLE : ce notebook ne fait QUE du feature engineering sans
# apprentissage. Aucune imputation, aucun One-Hot ici → ces
# opérations sont faites au NB03 (fit sur le train uniquement).
# Les NaN sont donc CONSERVÉS volontairement.
df = pd.read_csv("../data/processed/data_clean.csv", low_memory=False)

# Copie de travail
df_features = df.copy()

print("=" * 45)
print("DATASET CHARGÉ")
print("=" * 45)
print(f"  Lignes   : {df.shape[0]:,}")
print(f"  Colonnes : {df.shape[1]}")

# Rappel des NaN présents (normal, gérés au NB03)
nan_cols = df.isnull().sum()
nan_cols = nan_cols[nan_cols > 0]
print(f"\n  Colonnes avec NaN (imputées au NB03) : {len(nan_cols)}")
for col, n in nan_cols.items():
    print(f"    {col:<20} {n:>6,} ({n/len(df)*100:.1f}%)")

DATASET CHARGÉ
  Lignes   : 69,987
  Colonnes : 49

  Colonnes avec NaN (imputées au NB03) : 7
    race                  1,917 (2.7%)
    medical_specialty    33,652 (48.1%)
    diag_1                   10 (0.0%)
    diag_2                  293 (0.4%)
    diag_3                1,224 (1.7%)
    max_glu_serum        66,638 (95.2%)
    A1Cresult            57,141 (81.6%)


In [3]:
# =============================================================
# CELLULE 3 — ENCODAGE DE L'ÂGE
# =============================================================

# --- age_numeric : milieu de tranche (ordinal, info complète) ---
age_mapping = {
    "[0-10)"  : 5,  "[10-20)" : 15,
    "[20-30)" : 25, "[30-40)" : 35,
    "[40-50)" : 45, "[50-60)" : 55,
    "[60-70)" : 65, "[70-80)" : 75,
    "[80-90)" : 85, "[90-100)": 95,
}
# On crée une NOUVELLE colonne (on ne détruit pas 'age' d'origine)
df_features["age_numeric"] = df_features["age"].map(age_mapping)

# --- age_group : 3 groupes basés sur Strack et al. (Figure 2) ---
# [0-30) → 0 (jeunes) | [30-60) → 1 (adultes) | [60-100) → 2 (seniors)
def age_group(age_num):
    if age_num < 30:
        return 0
    elif age_num < 60:
        return 1
    else:
        return 2

df_features["age_group"] = df_features["age_numeric"].apply(age_group)

# On retire la colonne 'age' d'origine (remplacée par age_numeric + age_group)
df_features = df_features.drop(columns=["age"])

print("Encodage de l'âge :")
print(df_features[["age_numeric", "age_group"]].value_counts().sort_index())

Encodage de l'âge :
age_numeric  age_group
5            0              153
15           0              534
25           0             1121
35           1             2692
45           1             6828
55           1            12351
65           2            15688
75           2            17749
85           2            11110
95           2             1761
Name: count, dtype: int64


In [4]:
# =============================================================
# CELLULE 4 — ENCODAGE DES VARIABLES BINAIRES
# =============================================================

# Ces variables ont 2 valeurs → encodage 0/1 (déterministe, sans apprentissage)

# gender : Female → 0, Male → 1
df_features["gender"] = df_features["gender"].map({"Female": 0, "Male": 1})

# diabetesMed : No → 0, Yes → 1
df_features["diabetesMed"] = df_features["diabetesMed"].map({"No": 0, "Yes": 1})

# change : No → 0, Ch → 1
df_features["change"] = df_features["change"].map({"No": 0, "Ch": 1})

print("=" * 45)
print("ENCODAGE DES VARIABLES BINAIRES")
print("=" * 45)
for col in ["gender", "diabetesMed", "change"]:
    print(f"\n  {col} : {df_features[col].value_counts(dropna=False).to_dict()}")

ENCODAGE DES VARIABLES BINAIRES

  gender : {0: 37239, 1: 32748}

  diabetesMed : {1: 53302, 0: 16685}

  change : {0: 38492, 1: 31495}


In [5]:
# =============================================================
# CELLULE 5 — ENCODAGE ORDINAL DES MÉDICAMENTS
# =============================================================

# 4 valeurs ordonnées par intensité de traitement :
#   "No" → 0 | "Steady" → 1 | "Up" → 2 | "Down" → 3
# (mapping fixe, aucun apprentissage)
medicament_mapping = {"No": 0, "Steady": 1, "Up": 2, "Down": 3}

colonnes_medicaments = [
    "metformin", "repaglinide", "nateglinide", "chlorpropamide",
    "glimepiride", "acetohexamide", "glipizide", "glyburide",
    "tolbutamide", "pioglitazone", "rosiglitazone", "acarbose",
    "miglitol", "troglitazone", "tolazamide", "examide",
    "citoglipton", "insulin", "glyburide-metformin",
    "glipizide-metformin", "glimepiride-pioglitazone",
    "metformin-rosiglitazone", "metformin-pioglitazone",
]

for col in colonnes_medicaments:
    if col in df_features.columns:
        df_features[col] = df_features[col].map(medicament_mapping)

print("=" * 45)
print("ENCODAGE DES MÉDICAMENTS")
print("=" * 45)
print(f"  {len(colonnes_medicaments)} colonnes encodées (0=No, 1=Steady, 2=Up, 3=Down)")
print(f"\n  Exemple — insulin : {df_features['insulin'].value_counts().sort_index().to_dict()}")

ENCODAGE DES MÉDICAMENTS
  23 colonnes encodées (0=No, 1=Steady, 2=Up, 3=Down)

  Exemple — insulin : {0: 34265, 1: 21621, 2: 6777, 3: 7324}


In [6]:
# =============================================================
# CELLULE 6 — ENCODAGE ORDINAL DES TESTS A1C ET GLUCOSE
# =============================================================

# ATTENTION : on n'a PLUS "Non mesuré" (pas d'imputation au NB01).
# La valeur non-mesurée est un NaN. On encode les niveaux mesurés en
# ordinal, et on LAISSE le NaN tel quel (il sera imputé au NB03).
# Pour garder l'info "test fait ou non", on crée des flags en cellule 8.

# A1Cresult : Norm → 1, >7 → 2, >8 → 3  (NaN reste NaN)
A1C_mapping = {"Norm": 1, ">7": 2, ">8": 3}
df_features["A1Cresult"] = df_features["A1Cresult"].map(A1C_mapping)

# max_glu_serum : Norm → 1, >200 → 2, >300 → 3  (NaN reste NaN)
glu_mapping = {"Norm": 1, ">200": 2, ">300": 3}
df_features["max_glu_serum"] = df_features["max_glu_serum"].map(glu_mapping)

print("A1Cresult encodé (NaN = non mesuré, gardé pour le NB03) :")
print(df_features["A1Cresult"].value_counts(dropna=False).sort_index())
print("\nmax_glu_serum encodé :")
print(df_features["max_glu_serum"].value_counts(dropna=False).sort_index())

A1Cresult encodé (NaN = non mesuré, gardé pour le NB03) :
A1Cresult
1.0     3741
2.0     2866
3.0     6239
NaN    57141
Name: count, dtype: int64

max_glu_serum encodé :
max_glu_serum
1.0     1701
2.0      936
3.0      712
NaN    66638
Name: count, dtype: int64


In [7]:
# =============================================================
# CELLULE 7 — GROUPEMENT DES CODES ICD-9 EN 9 CHAPITRES
# =============================================================

# Règle FIXE basée sur Strack et al. 2014 (Table 2).
# On REMPLACE les codes par leur chapitre, mais on NE fait PAS
# le One-Hot ici → il sera fait au NB03 (avec alignement train/val/test).

def grouper_icd9(code):
    """Regroupe un code ICD-9 en chapitre clinique (règle fixe)."""
    if pd.isna(code):
        return "Autre"
    code = str(code)
    if code.startswith("E") or code.startswith("V"):
        return "Autre"
    if code.startswith("250"):
        return "Diabete"
    try:
        num = float(code)
    except ValueError:
        return "Autre"
    if (390 <= num <= 459) or num == 785:
        return "Circulatoire"
    elif (460 <= num <= 519) or num == 786:
        return "Respiratoire"
    elif (520 <= num <= 579) or num == 787:
        return "Digestif"
    elif (580 <= num <= 629) or num == 788:
        return "Genito_urinaire"
    elif 710 <= num <= 739:
        return "Musculo_squelettique"
    elif 140 <= num <= 239:
        return "Neoplasmes"
    elif 800 <= num <= 999:
        return "Blessure"
    else:
        return "Autre"

for col in ["diag_1", "diag_2", "diag_3"]:
    df_features[col] = df_features[col].apply(grouper_icd9)
    print(f"\n{col} regroupé :")
    print(df_features[col].value_counts())

print(f"\n✓ diag_1/2/3 → 9 chapitres (One-Hot au NB03)")


diag_1 regroupé :
diag_1
Circulatoire            21389
Autre                   12134
Respiratoire             9491
Digestif                 6488
Diabete                  5748
Blessure                 4694
Musculo_squelettique     4064
Genito_urinaire          3441
Neoplasmes               2538
Name: count, dtype: int64

diag_2 regroupé :
diag_2
Circulatoire            22083
Autre                   18373
Diabete                  9700
Respiratoire             6928
Genito_urinaire          5330
Digestif                 2856
Blessure                 1822
Neoplasmes               1600
Musculo_squelettique     1295
Name: count, dtype: int64

diag_3 regroupé :
diag_3
Autre                   21250
Circulatoire            20866
Diabete                 12547
Respiratoire             4652
Genito_urinaire          4049
Digestif                 2700
Blessure                 1409
Musculo_squelettique     1368
Neoplasmes               1146
Name: count, dtype: int64

✓ diag_1/2/3 → 9 chapitres (One-H

In [8]:
# =============================================================
# CELLULE 8 — CRÉATION DE NOUVELLES VARIABLES
# =============================================================

# --- 1. Score de risque hospitalier : total des visites antérieures ---
df_features["score_risque_hospitalier"] = (
    df_features["number_inpatient"]
    + df_features["number_emergency"]
    + df_features["number_outpatient"]
)

# --- 2. Ratio médicaments / procédures (+1 pour éviter /0) ---
df_features["ratio_medicaments_procedures"] = (
    df_features["num_medications"] / (df_features["num_procedures"] + 1)
).round(2)

# --- 3. Patient complexe : beaucoup de diagnostics ET de médicaments ---
df_features["patient_complexe"] = (
    (df_features["number_diagnoses"] > 5) & (df_features["num_medications"] > 15)
).astype(int)

# --- 4. Flags "test mesuré" (basés sur les NaN, AVANT toute imputation) ---
# Le papier Strack montre que le simple fait de mesurer l'A1c compte.
# Ces flags capturent l'info "test fait ou non" avant que le NB03 impute.
df_features["A1C_mesure"] = df_features["A1Cresult"].notna().astype(int)
df_features["glucose_mesure"] = df_features["max_glu_serum"].notna().astype(int)

nouvelles_vars = [
    "score_risque_hospitalier", "ratio_medicaments_procedures",
    "patient_complexe", "A1C_mesure", "glucose_mesure",
]
print("=" * 50)
print("NOUVELLES VARIABLES CRÉÉES")
print("=" * 50)
for var in nouvelles_vars:
    print(f"\n  {var} : {df_features[var].describe().to_dict()}")

NOUVELLES VARIABLES CRÉÉES

  score_risque_hospitalier : {'count': 69987.0, 'mean': 0.5597753868575592, 'std': 1.427771585938351, 'min': 0.0, '25%': 0.0, '50%': 0.0, '75%': 1.0, 'max': 49.0}

  ratio_medicaments_procedures : {'count': 69987.0, 'mean': 8.774284509980424, 'std': 5.545857837540936, 'min': 0.2, '25%': 4.5, '50%': 7.5, '75%': 12.0, 'max': 48.0}

  patient_complexe : {'count': 69987.0, 'mean': 0.36696815122808524, 'std': 0.4819811681836877, 'min': 0.0, '25%': 0.0, '50%': 0.0, '75%': 1.0, 'max': 1.0}

  A1C_mesure : {'count': 69987.0, 'mean': 0.18354837326932144, 'std': 0.3871182108885959, 'min': 0.0, '25%': 0.0, '50%': 0.0, '75%': 0.0, 'max': 1.0}

  glucose_mesure : {'count': 69987.0, 'mean': 0.04785174389529484, 'std': 0.2134539892265809, 'min': 0.0, '25%': 0.0, '50%': 0.0, '75%': 0.0, 'max': 1.0}


In [9]:
# =============================================================
# CELLULE 8bis — CONVERSION DES IDENTIFIANTS EN CATÉGORIEL
# =============================================================
# discharge_disposition_id, admission_type_id, admission_source_id
# sont des IDENTIFIANTS (codes), pas des quantités. Stockés en int64,
# ils seraient traités comme numériques par le NB03 (faux ordre).
# On les convertit en texte → ils seront one-hot encodés au NB03,
# chaque modalité traitée indépendamment.

ids_categoriels = ["discharge_disposition_id",
                   "admission_type_id",
                   "admission_source_id"]

for col in ids_categoriels:
    df_features[col] = df_features[col].astype(str)

print("=" * 45)
print("CONVERSION DES IDENTIFIANTS EN CATÉGORIEL")
print("=" * 45)
for col in ids_categoriels:
    print(f"  {col:<28} : {df_features[col].nunique()} modalités → catégoriel")

CONVERSION DES IDENTIFIANTS EN CATÉGORIEL
  discharge_disposition_id     : 21 modalités → catégoriel
  admission_type_id            : 8 modalités → catégoriel
  admission_source_id          : 17 modalités → catégoriel


In [10]:
# =============================================================
# CELLULE 9 — SUPPRESSION DES COLONNES CONSTANTES
# =============================================================

# examide, citoglipton (et parfois d'autres) sont toujours à 0 (jamais
# prescrits) → variance nulle → aucune information. On les retire.
const_cols = [c for c in df_features.columns if df_features[c].nunique() <= 1]

if const_cols:
    df_features = df_features.drop(columns=const_cols)
    print(f"✓ {len(const_cols)} colonnes constantes supprimées : {const_cols}")
else:
    print("✓ Aucune colonne constante à supprimer")

print(f"  Dimensions : {df_features.shape}")

✓ 3 colonnes constantes supprimées : ['examide', 'citoglipton', 'glimepiride-pioglitazone']
  Dimensions : (69987, 52)


In [11]:
# =============================================================
# CELLULE 10 — VÉRIFICATION FINALE
# =============================================================

print("=" * 55)
print("BILAN DU FEATURE ENGINEERING")
print("=" * 55)
print(f"\n  Lignes   : {df_features.shape[0]:,}")
print(f"  Colonnes : {df_features.shape[1]}")

# --- NaN restants (NORMAL : imputés au NB03) ---
print("\n" + "=" * 55)
print("VALEURS MANQUANTES (imputées au NB03, pas de leakage)")
print("=" * 55)
manquantes = df_features.isnull().sum()
manquantes = manquantes[manquantes > 0]
if len(manquantes) == 0:
    print("  Aucune.")
else:
    for col, val in manquantes.items():
        print(f"  {col:<20} {val:>6,} ({val/len(df_features)*100:.1f}%)")

# --- Colonnes catégorielles restantes (One-Hot au NB03) ---
print("\n" + "=" * 55)
print("COLONNES CATÉGORIELLES RESTANTES (One-Hot au NB03)")
print("=" * 55)
cat_cols = df_features.select_dtypes(include=["object"]).columns.tolist()
# On retire les identifiants et la cible brute de l'affichage
cat_cols = [c for c in cat_cols if c not in ["readmitted"]]
for col in cat_cols:
    print(f"  {col:<20} : {df_features[col].nunique()} modalités")

print("\n" + "=" * 55)
print("APERÇU DU DATASET")
print("=" * 55)
display(df_features.head())

BILAN DU FEATURE ENGINEERING

  Lignes   : 69,987
  Colonnes : 52

VALEURS MANQUANTES (imputées au NB03, pas de leakage)
  race                  1,917 (2.7%)
  medical_specialty    33,652 (48.1%)
  max_glu_serum        66,638 (95.2%)
  A1Cresult            57,141 (81.6%)

COLONNES CATÉGORIELLES RESTANTES (One-Hot au NB03)
  race                 : 5 modalités
  admission_type_id    : 8 modalités
  discharge_disposition_id : 21 modalités
  admission_source_id  : 17 modalités
  medical_specialty    : 70 modalités
  diag_1               : 9 modalités
  diag_2               : 9 modalités
  diag_3               : 9 modalités

APERÇU DU DATASET


C:\Users\malek\AppData\Local\Temp\ipykernel_19324\4234688074.py:27: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df_features.select_dtypes(include=["object"]).columns.tolist()


,encounter_id,patient_nbr,race,gender,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,insulin,glyburide-metformin,glipizide-metformin,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted,readmitted_binary,age_numeric,age_group,score_risque_hospitalier,ratio_medicaments_procedures,patient_complexe,A1C_mesure,glucose_mesure
0,12522,48330783,Caucasian,0,2,1,4,13,NaN,68,2,28,0,0,0,Circulatoire,Circulatoire,Autre,8,NaN,NaN,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,1,NO,0,85,2,0,9.33,1,0,0
1,15738,63555939,Caucasian,0,3,3,4,12,InternalMedicine,33,3,18,0,0,0,Circulatoire,Neoplasmes,Respiratoire,8,NaN,NaN,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,1,NO,0,95,2,0,4.50,1,0,0
2,16680,42519267,Caucasian,1,1,1,7,1,NaN,51,0,8,0,0,0,Neoplasmes,Neoplasmes,Diabete,5,NaN,NaN,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,1,NO,0,45,1,0,8.00,0,0,0
3,28236,89869032,AfricanAmerican,0,1,1,7,9,NaN,47,2,17,0,0,0,Diabete,Circulatoire,Blessure,9,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,>30,0,45,1,0,5.67,1,0,0
4,35754,82637451,Caucasian,1,2,1,2,3,NaN,31,6,16,0,0,0,Circulatoire,Circulatoire,Diabete,9,NaN,NaN,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,>30,0,55,1,0,2.29,1,0,0


In [ ]:
# =============================================================
# CELLULE 11 — SAUVEGARDE DU DATASET FEATURE-ENGINEERED
# =============================================================

# Sauvegarde
chemin = "../data/processed/data_features.csv"
df_features.to_csv(chemin, index=False)



print("✓ Dataset sauvegardé avec succès.")
print(f"  Chemin   : {chemin}")
print(f"  Lignes   : {df_features.shape[0]:,}")
print(f"  Colonnes : {df_features.shape[1]}")
print(f"\n  CONSERVÉ pour le NB03 :")
print(f"    - encounter_id, patient_nbr (split + anti-leakage)")
print(f"    - readmitted (brute) + readmitted_binary (cible)")
print(f"    - colonnes catégorielles en texte (diag, race, specialty…)")
print(f"    - NaN non imputés (A1Cresult, max_glu_serum, medical_specialty…)")
print("\n➡️  Prochaine étape : NB03 — Split + Preprocessing")
print("     (imputation + scaling + One-Hot, fit sur train uniquement)")

✓ Dataset sauvegardé avec succès.
  Chemin   : ../data/processed/data_features.csv
  Lignes   : 69,987
  Colonnes : 52

  CONSERVÉ pour le NB03 :
    - encounter_id, patient_nbr (split + anti-leakage)
    - readmitted (brute) + readmitted_binary (cible)
    - colonnes catégorielles en texte (diag, race, specialty…)
    - NaN non imputés (A1Cresult, max_glu_serum, medical_specialty…)

➡️  Prochaine étape : NB03 — Split + Preprocessing
     (imputation + scaling + One-Hot, fit sur train uniquement)


# NB03 — Feature Engineering

**Projet :** Prédiction de la réhospitalisation précoce des patients diabétiques

---

## Objectif
Créer et transformer les variables via des **règles fixes, sans apprentissage**.
Toutes les transformations sont déterministes → aucune statistique n'est calculée
sur les données, donc **aucun risque de leakage** même avant le split.

## Transformations réalisées (déterministes)
| Variable | Transformation |
|---|---|
| `age` | `age_numeric` (milieu de tranche) + `age_group` (3 catégories, Strack Fig. 2) |
| `gender`, `diabetesMed`, `change` | Encodage binaire (0/1) |
| 23 médicaments | Encodage **ordinal** (0=No, 1=Steady, 2=Up, 3=Down) |
| `A1Cresult`, `max_glu_serum` | Encodage **ordinal** (1→3), NaN conservé |
| `diag_1/2/3` | Groupement en **9 chapitres ICD-9** (Strack Table 2) |

## Nouvelles variables créées
| Variable | Description |
|---|---|
| `score_risque_hospitalier` | Somme des visites antérieures (outpatient + emergency + inpatient) |
| `ratio_medicaments_procedures` | `num_medications / (num_procedures + 1)` |
| `patient_complexe` | 1 si `number_diagnoses > 5` ET `num_medications > 15` |
| `A1C_mesure`, `glucose_mesure` | Flag "test effectué ou non" (basé sur les NaN) |

## Choix méthodologiques
- **Encodage ordinal** (médicaments, tests) plutôt que One-Hot : préserve l'ordre
  naturel (intensité) et réduit fortement la dimensionnalité.
- **"Non mesuré" = NaN** (pas 0) : sépare l'info "test fait" (flag dédié) du niveau
  du résultat, cliniquement plus juste (cf. hypothèse centrale de Strack et al.).
- **Suppression des colonnes constantes** (`examide`, `citoglipton`) : variance nulle,
  aucune information.

## Reporté au NB03 (anti-leakage)
- Imputation des NaN, One-Hot de `diag`/`race`/`medical_specialty`, scaling
  → tout ce qui **apprend** une statistique se fait après le split, sur le train.

## Entrée / Sortie
- **Entrée** : `data/processed/data_clean.csv`
- **Sortie** : `data/processed/data_features.csv` (colonnes catégorielles et NaN conservés)
